# Parameter Fitting: Recovering Inputs from Data

Notebook 04 drove a case toward a target chosen by the engineer. Here the target is
data: a set of measurements exists, and the question is which parameters could have
produced them. It is the same optimization machinery pointed the other way round,
which is why parameter fitting and design optimization share an API in `uqtopus`.

Two things change from notebook 04. There are now two unknowns instead of one, and
the objective is a vector of per-sensor mismatches rather than a single number. The
measurements are synthetic and deliberately noisy, so the true answer is known and
the quality of the fit can actually be judged.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

import uqtopus as uqt

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

## 1. A template with two free parameters

`templates/scalarTransportFit` is the `scalarTransportFoam` case of notebook 01 with
one extra placeholder. The diffusivity was already exposed; the scalar value imposed
at the inlet now is too:

| parameter key | file | placeholder |
|---|---|---|
| `constant__transportProperties__DT` | `constant/transportProperties` | `{{DT}}` |
| `0__T__Tin` | `0/T` | `{{Tin}}` |

The key names are the path to the file plus the placeholder name, joined by double
underscores, which is the same convention used throughout the earlier notebooks. A
file is only rendered when one of its placeholders appears in `params`, so the
second placeholder lives in its own template folder rather than being added to the
one that notebooks 01 and 02 use.

In [ ]:
DT_KEY  = 'constant__transportProperties__DT'
TIN_KEY = '0__T__Tin'

sim = uqt.OpenFOAMSimulator(
    template_path="templates/scalarTransportFit",
    solver_script="Allrun",
    output_path="experiments/fitting",
    qoi_variables=["T"],
    qoi_times=["0.1"]
)
sim

## 2. Synthetic measurements

A real fit starts from an experiment. Standing in for one, a single run at known
parameter values provides the ground truth, a scatter of cells plays the part of
sensor locations, and Gaussian noise plays the part of measurement error.

Sampling a subset of cells rather than the whole field matters more than it looks.
Real data comes from a handful of probes, and a fit that only works when every cell
is observed is not a fit anyone can reproduce in a laboratory.

In [ ]:
DT_TRUE, TIN_TRUE = 0.08, 1.4
N_SENSORS, NOISE = 150, 0.02

ds_true = sim.run({DT_KEY: DT_TRUE, TIN_KEY: TIN_TRUE})
T_true = ds_true['T'].values[-1]

rng = np.random.default_rng(42)
sensors = rng.choice(T_true.size, size=N_SENSORS, replace=False)
measured = T_true[sensors] + rng.normal(0, NOISE, N_SENSORS)

print(f"cells in the case  {T_true.size}")
print(f"sensors            {N_SENSORS}")
print(f"noise std          {NOISE}  ({100 * NOISE / TIN_TRUE:.1f}% of the inlet value)")

## 3. Why a residual and not a scalar

`DT` and `Tin` act on the solution in different ways. The inlet value scales the
whole field, since the transport equation is linear in `T`, while the diffusivity
changes its shape. That difference is what makes both recoverable at once.

Collapsing the comparison into a single number would throw it away: many
combinations of a larger `Tin` and a smaller `DT` produce the same mean, and the
optimizer would have no way to tell them apart. `as_residual` hands
`least_squares` the full vector of per-sensor mismatches instead, so the shape
information survives and the two parameters separate.

In [ ]:
def residual_fn(ds):
    return ds['T'].values[-1][sensors] - measured

r = sim.as_residual(residual_fn=residual_fn, param_keys=[DT_KEY, TIN_KEY])

sim.reset()
result = least_squares(
    r,
    x0=[0.15, 1.0],                       # deliberately away from the truth
    bounds=([0.01, 0.5], [0.3, 3.0])
)

DT_fit, TIN_fit = result.x
print(f"           true      fitted     error")
print(f"DT         {DT_TRUE:<9.4f} {DT_fit:<10.4f} {abs(DT_fit - DT_TRUE) / DT_TRUE:.2%}")
print(f"Tin        {TIN_TRUE:<9.4f} {TIN_fit:<10.4f} {abs(TIN_fit - TIN_TRUE) / TIN_TRUE:.2%}")
print(f"\nsimulations {sim.run_count}")

## 4. How good is the fit

The left panel compares what the fitted parameters predict at each sensor against
what was measured there; a perfect fit would put every point on the diagonal. The
right panel shows the residuals that remain.

The useful check is not that the residuals are small but that they look like the
noise that was added: scattered around zero with no structure. Residuals that curve
or fan out mean the parameters are compensating for something the model is getting
wrong, and a tighter fit would only hide it.

In [ ]:
predicted = sim.run({DT_KEY: DT_fit, TIN_KEY: TIN_fit})['T'].values[-1][sensors]
residuals = predicted - measured

fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), constrained_layout=True)

lims = [min(measured.min(), predicted.min()), max(measured.max(), predicted.max())]
axs[0].plot(lims, lims, '-', color='k', linewidth=1)
axs[0].plot(measured, predicted, 'o', color='tab:gray', alpha=0.6, markersize=4)
axs[0].set_xlabel('measured T')
axs[0].set_ylabel('fitted T')

axs[1].axhline(0, color='k', linewidth=1)
axs[1].axhline(NOISE, color='k', linestyle='--', linewidth=0.8)
axs[1].axhline(-NOISE, color='k', linestyle='--', linewidth=0.8)
axs[1].plot(measured, residuals, 'o', color='tab:gray', alpha=0.6, markersize=4)
axs[1].set_xlabel('measured T')
axs[1].set_ylabel('residual')

plt.show()

print(f"residual std   {residuals.std():.4f}")
print(f"noise added    {NOISE:.4f}")

A residual spread that lands near the noise level means the fit has extracted about
as much as the data contains. Driving it below that would be fitting the noise.

The same call works unchanged on a case with more parameters, provided the data
constrains them. Notebook 03 covers the other way to make a study like this
affordable: fitting an emulator once, then optimizing against it instead of against
the solver.